**Author:** Dorys Trujillo  
**Project:** Legal Uncertainty Index (IIJ)   
**Data Source:** Ministry of Commerce, Industry and Tourism (MinCIT)  
**Period:** 2018–2025

## Natural Language Preprocessing

In [1]:
# Imports
from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path
from typing import Iterable

import pandas as pd
import spacy
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

C:\Users\dtruj\Documents\venv_nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# define project root dynamically (assumes execution inside notebooks/)
PROJECT_ROOT = Path().resolve().parent

# input: corpus organized by regulation
OCR_ROOT = PROJECT_ROOT / "data_processed" / "txt_normative_rag"

# output: cleaned corpus
CLEAN_ROOT = PROJECT_ROOT / "data_processed" / "corpus_clean"

# manifests: global tracking files
MANIFEST_ROOT = PROJECT_ROOT / "manifests"

# create output directories if they do not exist
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

from pathlib import Path

# define path correctly relative to project root
PROJECT_ROOT = Path().resolve().parents[0]

src_path = PROJECT_ROOT / "src_python"
src_path.mkdir(parents=True, exist_ok=True)

init_path = src_path / "__init__.py"
init_path.touch(exist_ok=True)

# sanity checks
#print("project_root:", PROJECT_ROOT)
#print("ocr_root exists:", OCR_ROOT.exists(), OCR_ROOT)
#print("clean_root:", CLEAN_ROOT)
#print("manifest_root:", MANIFEST_ROOT)

# discover all txt files recursively (preserving folder structure by regulation)
ocr_files = sorted(OCR_ROOT.rglob("*.txt"))

print(f"ocr files found: {len(ocr_files)}")

# We build basic inventory
rows = []
for txt_path in ocr_files:
    rel_path = txt_path.relative_to(OCR_ROOT)
    content = txt_path.read_text(encoding="utf-8", errors="ignore")
    
    rows.append({
        "ocr_file": str(txt_path),
        "rel_path": str(rel_path),
        "n_chars": len(content),
        "n_lines": content.count("\n") + 1,
        "is_empty": len(content.strip()) == 0
    })

df_ocr_inventory = pd.DataFrame(rows)

print("\nQuick summary:")
print(df_ocr_inventory["is_empty"].value_counts(dropna=False))

display(df_ocr_inventory.head(10))

ocr files found: 333

Quick summary:
is_empty
False    333
Name: count, dtype: int64


,ocr_file,rel_path,n_chars,n_lines,is_empty
0,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Aplicación de derechos compensatorios\id_0085.txt,83069,1287,False
1,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0008.txt,3116,58,False
2,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0019.txt,4460,76,False
3,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0026.txt,4947,120,False
4,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0030.txt,2482,53,False
5,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0039.txt,5116,104,False
6,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0050.txt,4414,66,False
7,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0051.txt,5963,116,False
8,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0055.txt,6458,131,False
9,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0067.txt,11020,247,False


#### 1. Basic Cleaning Stage
This step performs basic text preprocessing on the OCR corpus. It cleans and normalizes the text (line breaks, spaces, Unicode, and hyphenated words), preserves the original folder structure by regulation, saves the cleaned files in a new processing stage, and generates a manifest to track transformations and ensure traceability.

In [3]:
# subfolder for this stage
BASIC_CLEAN_ROOT = CLEAN_ROOT / "basic_clean"
BASIC_CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

BASIC_MANIFEST_PATH = MANIFEST_ROOT / "manifest_basic_clean.csv"

# basic text cleaning function
def basic_clean_text(text: str) -> str:
    # 1) normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) replace invisible characters and unusual spaces
    text = text.replace("\ufeff", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\x0c", "\n")

    # 3) replace common typographic ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")

    # 4) normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # 5) join words split by hyphen at line breaks
    # example: econo-\nmía -> economía
    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)

    # 6) collapse repeated spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # 7) trim spaces at the beginning and end of lines
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)

    # 8) reduce multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 9) final strip
    text = text.strip()

    return text

# process all txt files
rows = []

for txt_path in ocr_files:
    rel_path = txt_path.relative_to(OCR_ROOT)
    out_path = BASIC_CLEAN_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    clean_text = basic_clean_text(raw_text)

    out_path.write_text(clean_text, encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "raw_chars": len(raw_text),
        "clean_chars": len(clean_text),
        "char_diff": len(raw_text) - len(clean_text),
        "raw_lines": raw_text.count("\n") + 1,
        "clean_lines": clean_text.count("\n") + 1,
        "is_empty_after_clean": len(clean_text.strip()) == 0,
        "status": "ok"
    })

df_basic_clean = pd.DataFrame(rows)
df_basic_clean.to_csv(BASIC_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_basic_clean))
print("empty after cleaning:", df_basic_clean["is_empty_after_clean"].sum())
print("manifest saved to:", BASIC_MANIFEST_PATH)

#display(df_basic_clean.head(10))

files processed: 333
empty after cleaning: 0
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_basic_clean.csv


In [4]:
#Check that the character reduction is reasonable
#df_basic_clean[["raw_chars", "clean_chars", "char_diff"]].describe()

### 2. Structural Normalization

This stage performs structural normalization of the cleaned corpus by identifying and extracting the most relevant legal content. It detects key sections such as the decree title, action clauses (e.g., “decreta”), and the first article, applying a prioritized strategy to retain meaningful text. The process reduces noise, standardizes formatting, preserves the original folder structure, and generates a manifest to track transformations and applied strategies.

In [5]:
# subfolder for this stage
NORMALIZED_ROOT = CLEAN_ROOT / "normalized"
NORMALIZED_ROOT.mkdir(parents=True, exist_ok=True)

NORMALIZED_MANIFEST_PATH = MANIFEST_ROOT / "manifest_normalized.csv"

# structural patterns
TITLE_START_PATTERNS = [
    r"por el cual",
    r"por la cual",
    r"por medio del cual",
    r"por medio de la cual",
]

PRESIDENT_MARKERS = [
    r"el presidente de la república",
    r"el presidente de la republica",
    r"la ministra de",
    r"el ministro de",
]

ACTION_PATTERNS = [
    r"\bdecreta\s*:?\b",
    r"\bresuelve\s*:?\b",
]

ARTICLE_START_PATTERNS = [
    r"art[ií]culo\s+1\b",
    r"art[ií]culo\s+primero\b",
]

# compile regex patterns once
TITLE_START_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in TITLE_START_PATTERNS]
PRESIDENT_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in PRESIDENT_MARKERS]
ACTION_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ACTION_PATTERNS]
ARTICLE_START_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ARTICLE_START_PATTERNS]


def find_first_match_position(text: str, patterns: list[re.Pattern], start_pos: int = 0):
    """return the first match position among multiple compiled patterns."""
    tail = text[start_pos:]
    positions = []

    for pattern in patterns:
        match = pattern.search(tail)
        if match:
            positions.append(start_pos + match.start())

    return min(positions) if positions else None


def find_title_start(text: str, search_window: int = 4000):
    """find the beginning of the decree title within the first part of the document."""
    head = text[:search_window]
    positions = []

    for pattern in TITLE_START_REGEX:
        match = pattern.search(head)
        if match:
            positions.append(match.start())

    return min(positions) if positions else None


def extract_title_block(text: str):
    """extract the decree title block using title, authority, and action markers."""
    start = find_title_start(text)
    if start is None:
        return None

    president_pos = find_first_match_position(text, PRESIDENT_REGEX, start_pos=start)
    action_pos = find_first_match_position(text, ACTION_REGEX)

    candidates = [pos for pos in [president_pos, action_pos] if pos is not None and pos > start]

    if candidates:
        end = min(candidates)
        title = text[start:end].strip()
    else:
        title = text[start:start + 1200].strip()

    title = title.strip('“”"\' \n\t')
    title = re.sub(r"\s+", " ", title)

    return title if len(title) > 20 else None


def normalize_text_structural(text: str):
    """apply structural normalization and keep the most relevant legal content."""
    text = text.lower()
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    title_block = extract_title_block(text)
    action_pos = find_first_match_position(text, ACTION_REGEX)
    article_pos = find_first_match_position(text, ARTICLE_START_REGEX)

    # priority 1: title block + action block
    if title_block and action_pos is not None:
        action_block = text[action_pos:].strip()
        text = f"{title_block}\n\n{action_block}"
        strategy = "title_block_plus_action"

    # priority 2: title block + first article
    elif title_block and article_pos is not None:
        article_block = text[article_pos:].strip()
        text = f"{title_block}\n\n{article_block}"
        strategy = "title_block_plus_article"

    # fallback 1: action block only
    elif action_pos is not None:
        text = text[action_pos:].strip()
        strategy = "action_only"

    # fallback 2: first article only
    elif article_pos is not None:
        text = text[article_pos:].strip()
        strategy = "article_only"

    # fallback 3: no structural cut
    else:
        strategy = "fallback_no_structural_cut"

    # residual cleanup
    text = re.sub(r"\b(página|pagina)\s*\d+\b", " ", text)
    text = re.sub(r"-\s*\d+\s*-", " ", text)
    text = re.sub(r"#{2,}\s*page\s*\d+", " ", text)

    lines = text.split("\n")
    lines = [line.strip() for line in lines if len(line.strip()) > 2]
    text = "\n".join(lines)

    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip(), strategy


# process files from basic cleaning stage
rows = []

basic_files = sorted(BASIC_CLEAN_ROOT.rglob("*.txt"))

for txt_path in basic_files:
    rel_path = txt_path.relative_to(BASIC_CLEAN_ROOT)
    out_path = NORMALIZED_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    norm_text, strategy = normalize_text_structural(text)

    out_path.write_text(norm_text, encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "chars_before": len(text),
        "chars_after": len(norm_text),
        "char_diff": len(text) - len(norm_text),
        "lines_before": text.count("\n") + 1,
        "lines_after": norm_text.count("\n") + 1,
        "strategy": strategy,
        "status": "ok"
    })

df_normalized = pd.DataFrame(rows)
df_normalized.to_csv(NORMALIZED_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_normalized))
print("manifest saved to:", NORMALIZED_MANIFEST_PATH)
print("\nstrategies applied:")
print(df_normalized["strategy"].value_counts())

display(df_normalized[["chars_before", "chars_after", "char_diff"]].describe())

files processed: 333
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_normalized.csv

strategies applied:
strategy
title_block_plus_action     332
title_block_plus_article      1
Name: count, dtype: int64


,chars_before,chars_after,char_diff
count,333.000000,333.000000,333.000000
mean,18492.831832,12789.306306,5703.525526
std,22468.612980,21877.476624,3734.504555
min,1443.000000,487.000000,723.000000
25%,6200.000000,1790.000000,3450.000000
50%,10564.000000,4330.000000,4870.000000
75%,21409.000000,13577.000000,7035.000000
max,150286.000000,146256.000000,35141.000000


### 3. NLP-Oriented Cleaning

This stage applies NLP-oriented cleaning rules to reduce residual noise in the normalized corpus. It preserves key legal structure such as the title and action line, removes headers, table-like artifacts, and low-semantic content, and generates a manifest to track the transformation results.

In [6]:
# output
NLP_CLEAN_ROOT = CLEAN_ROOT / "nlp_clean"
NORMALIZED_ROOT = CLEAN_ROOT / "normalized"
NLP_CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

NLP_MANIFEST_PATH = MANIFEST_ROOT / "manifest_nlp_clean.csv"

# patterns
HEADER_PATTERNS = [
    r"continuaci[oó]n del decreto",
    r"decreto n[uú]mero de hoja[nm]?\s*\.?\s*\d+",
    r"decreto n[uú]mero de\s*.*hoja.*",
    r"p[aá]gina\s*\d+",
    r"gd-fm-\d+\s*v\d+",
    r"rep[uú]blica de colombia",
    r"ministerio de .*",
    r"por el cual se adiciona el cap[ií]tulo",
    r"decreto numero de hoja",
    r"decreto n[uú]mero de hoja",
    r"hoja n",
    r"gd-fm-\d+",
    r"gn-fm-\d+",
    r"an-fa-m\d+",
    r"cn_er_a\d+",
]

TABLE_SYMBOL_PATTERN = r"[|\[\]{}<>]"
LONG_NUMERIC_CODE_PATTERN = r"\b\d{5,}\b"

ACTION_LINE_PATTERNS = [
    r"^decreta\s*:?\s*$",
    r"^resuelve\s*:?\s*$",
]

LEGAL_WHITELIST = {
    "decreto", "decretos", "ley", "leyes", "articulo", "artículos", "articulo",
    "articulos", "paragrafo", "parágrafos", "paragrafos", "capitulo", "capítulos",
    "capitulos", "titulo", "títulos", "titulos", "seccion", "secciones", "parte",
    "partes", "libro", "libros", "numeral", "numerales", "inciso", "incisos",
    "vigencia", "deroga", "derogan", "modifica", "modifican", "adiciona",
    "adicionan", "sustituye", "sustituyen", "reglamenta", "reglamentan",
    "ministerio", "república", "republica", "colombia", "resolucion", "resolución",
    "publicacion", "publicación", "objeto", "ámbito", "ambito", "aplicacion",
    "aplicación", "registro", "nacional", "turismo", "importacion", "importación",
    "exportacion", "exportación", "arancel", "arancelaria", "arancelarias",
    "subpartida", "subpartidas", "licencia", "previa", "control", "obligacion",
    "obligaciones", "servicios", "prestadores", "plataformas", "digitales",
    "operadores", "comercio", "industria", "turismo", "presente"
}

SHORT_VALID = {
    "de", "la", "el", "en", "al", "lo", "se", "un", "no", "ya", "del", "las", "los"
}

# compile patterns once
HEADER_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in HEADER_PATTERNS]
ACTION_LINE_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ACTION_LINE_PATTERNS]
TABLE_SYMBOL_REGEX = re.compile(TABLE_SYMBOL_PATTERN)
LONG_NUMERIC_CODE_REGEX = re.compile(LONG_NUMERIC_CODE_PATTERN)

# token rules
def has_vowel(token: str) -> bool:
    return bool(re.search(r"[aeiouáéíóúü]", token))

def max_consecutive_consonants(token: str) -> int:
    consonant_groups = re.findall(r"[^aeiouáéíóúü\W\d_]+", token.lower())
    return max((len(group) for group in consonant_groups), default=0)

def vowel_ratio(token: str) -> float:
    letters = sum(ch.isalpha() for ch in token)
    vowels = sum(ch in "aeiouáéíóúü" for ch in token.lower())
    return vowels / letters if letters > 0 else 0.0

def normalize_token(token: str) -> str:
    return token.strip().lower()

def is_valid_token(token: str) -> bool:
    token = normalize_token(token)

    if not token:
        return False

    if token in LEGAL_WHITELIST:
        return True

    if re.fullmatch(r"\d+", token):
        return False

    if len(token) == 1:
        return False

    if len(token) == 2:
        return token in SHORT_VALID

    if re.search(r"\d", token):
        return False

    if not has_vowel(token):
        return False

    if max_consecutive_consonants(token) >= 4:
        return False

    if len(token) >= 5 and vowel_ratio(token) < 0.30:
        return False

    if len(token) >= 5 and len(set(token)) <= 3:   #new rule: low character diversity (OCR noise)
        return False

    if "q" in token and "qu" not in token:  # rule for q and qu (more strict)
        return False

    if re.search(r"(.)\1{2,}", token):
        return False

    if len(token) >= 6 and len(set(token)) <= 3:
        return False

    if (
        re.fullmatch(r"[a-záéíóúüñ]{2,4}", token)
        and token not in SHORT_VALID
        and token not in LEGAL_WHITELIST
    ):
        if vowel_ratio(token) < 0.34:
            return False

    return True

# line rules
def is_action_line(line: str) -> bool:
    line_norm = line.strip().lower()
    return any(pattern.search(line_norm) for pattern in ACTION_LINE_REGEX)

def is_header_or_footer_line(line: str) -> bool:
    line_norm = line.strip().lower()
    if not line_norm:
        return False
    return any(pattern.search(line_norm) for pattern in HEADER_REGEX)

def is_table_noise_line(line: str) -> bool:
    line_norm = line.strip().lower()
    if not line_norm:
        return False

    if len(TABLE_SYMBOL_REGEX.findall(line_norm)) >= 2:
        return True

    if LONG_NUMERIC_CODE_REGEX.search(line_norm):
        return True

    if re.search(r"(\.\s*){3,}", line_norm):
        return True

    tokens = re.findall(r"[a-záéíóúüñ0-9]+", line_norm)
    if not tokens:
        return False

    digits = sum(ch.isdigit() for ch in line_norm)
    letters = sum(ch.isalpha() for ch in line_norm)
    if digits > 0 and digits >= letters:
        return True

    short_tokens = sum(1 for token in tokens if len(token) <= 2)
    if len(tokens) >= 5 and short_tokens / len(tokens) >= 0.6:
        return True

    invalid_tokens = sum(1 for token in tokens if not is_valid_token(token))
    if len(tokens) >= 4 and invalid_tokens / len(tokens) >= 0.6:
        return True

    return False

# protected block
def split_protected_block(text: str):
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    if not lines:
        return "", "", []

    title_line = lines[0]
    action_line = ""
    rest_start = 1

    if len(lines) > 1 and is_action_line(lines[1]):
        action_line = lines[1]
        rest_start = 2

    rest_lines = lines[rest_start:]
    return title_line, action_line, rest_lines


# residual cleaning
def clean_residual_text(lines: list[str]) -> tuple[str, dict]:
    kept_lines = []
    removed_header_footer = 0
    removed_table_noise = 0
    removed_low_semantic = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue

        tokens_preview = re.findall(r"[a-záéíóúüñ0-9]+", line.lower())

        if len(tokens_preview) <= 2:
            valid_preview = [t for t in tokens_preview if is_valid_token(t)]
            if len(valid_preview) == 0:
                removed_low_semantic += 1
                continue

        if is_header_or_footer_line(line):
            removed_header_footer += 1
            continue

        if is_table_noise_line(line):
            removed_table_noise += 1
            continue

        line = line.lower()
        line = re.sub(r"(\.\s*){3,}", " ", line)
        line = re.sub(r"\.{2,}", " ", line)
        line = re.sub(r"[^a-záéíóúüñ0-9\s\.,;:()\-\n]", " ", line)
        line = re.sub(r"\(\s*\d+\s*\)", " ", line)
        line = re.sub(r"\b\d+\b", " ", line) # remove standalone numeric tokens (e.g., article numbers, decree IDs, codes)
        line = re.sub(r"[.,;:()\-_\/]+", " ", line)
        line = re.sub(r"\s+", " ", line).strip()

        if not line:
            removed_low_semantic += 1
            continue

        tokens = line.split()
        valid_tokens = [token for token in tokens if is_valid_token(token)]

        if len(valid_tokens) == 0:
            removed_low_semantic += 1
            continue

        if len(tokens) >= 4 and len(valid_tokens) / len(tokens) < 0.5:
            removed_low_semantic += 1
            continue

        kept_lines.append(" ".join(valid_tokens))

    # remove duplicated lines while preserving order
    seen_lines = set()
    deduped_lines = []

    for line in kept_lines:
        if line not in seen_lines:
            deduped_lines.append(line)
            seen_lines.add(line)

    cleaned = "\n".join(deduped_lines)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()

    stats = {
        "removed_header_footer_lines": removed_header_footer,
        "removed_table_noise_lines": removed_table_noise,
        "removed_low_semantic_lines": removed_low_semantic,
    }

    return cleaned, stats

# main cleaner
def nlp_clean_text(text: str) -> tuple[str, dict]:
    title_line, action_line, rest_lines = split_protected_block(text)

    title_line_clean = re.sub(r"\s+", " ", title_line.lower().strip())
    action_line_clean = re.sub(r"\s+", " ", action_line.lower().strip())

    residual_text, stats = clean_residual_text(rest_lines)

    parts = []
    if title_line_clean:
        parts.append(title_line_clean)
    if action_line_clean:
        parts.append(action_line_clean)
    if residual_text:
        parts.append(residual_text)

    final_text = "\n".join(parts).strip()

    stats["protected_title_kept"] = bool(title_line_clean)
    stats["protected_action_kept"] = bool(action_line_clean)

    return final_text, stats

# process files from structural normalization stage
rows = []

normalized_files = sorted(NORMALIZED_ROOT.rglob("*.txt"))

for txt_path in normalized_files:
    rel_path = txt_path.relative_to(NORMALIZED_ROOT)
    out_path = NLP_CLEAN_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    clean_text, stats = nlp_clean_text(text)

    out_path.write_text(clean_text, encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "chars_before": len(text),
        "chars_after": len(clean_text),
        "char_diff": len(text) - len(clean_text),
        "lines_before": text.count("\n") + 1,
        "lines_after": clean_text.count("\n") + 1,
        "removed_header_footer_lines": stats["removed_header_footer_lines"],
        "removed_table_noise_lines": stats["removed_table_noise_lines"],
        "removed_low_semantic_lines": stats["removed_low_semantic_lines"],
        "protected_title_kept": stats["protected_title_kept"],
        "protected_action_kept": stats["protected_action_kept"],
        "is_empty_after_clean": len(clean_text.strip()) == 0,
        "status": "ok"
    })

df_nlp_clean = pd.DataFrame(rows)
df_nlp_clean.to_csv(NLP_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_nlp_clean))
print("empty after cleaning:", df_nlp_clean["is_empty_after_clean"].sum())
print("manifest saved to:", NLP_MANIFEST_PATH)

display(
    df_nlp_clean[
        [
            "chars_before",
            "chars_after",
            "char_diff",
            "removed_header_footer_lines",
            "removed_table_noise_lines",
            "removed_low_semantic_lines"
        ]
    ].describe()
)

files processed: 333
empty after cleaning: 0
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_nlp_clean.csv


,chars_before,chars_after,char_diff,removed_header_footer_lines,removed_table_noise_lines,removed_low_semantic_lines
count,333.000000,333.000000,333.000000,333.000000,333.000000,333.000000
mean,12789.306306,9557.957958,3231.348348,10.654655,11.387387,1.756757
std,21877.476624,16746.906185,5542.727192,17.463670,21.140453,5.789297
min,487.000000,244.000000,63.000000,0.000000,0.000000,0.000000
25%,1790.000000,1282.000000,422.000000,1.000000,1.000000,0.000000
50%,4330.000000,3293.000000,1083.000000,4.000000,4.000000,0.000000
75%,13577.000000,10102.000000,3501.000000,11.000000,12.000000,1.000000
max,146256.000000,101166.000000,45866.000000,116.000000,143.000000,59.000000


### 4. Tokenization
This stage performs tokenization of the cleaned corpus by splitting the text into individual words based on a defined pattern. It removes very short tokens, preserves the folder structure, stores tokens as one per line, and generates a manifest with basic statistics for traceability.

In [7]:
# input from previous stage
NLP_CLEAN_ROOT = CLEAN_ROOT / "nlp_clean"

# output
TOKENS_ROOT = CLEAN_ROOT / "tokens"
TOKENS_ROOT.mkdir(parents=True, exist_ok=True)

TOKENS_MANIFEST_PATH = MANIFEST_ROOT / "manifest_tokens.csv"

# token pattern
TOKEN_PATTERN = r"[a-zñ]+"

def remove_accents(text: str) -> str:
    return "".join(
        char for char in unicodedata.normalize("NFD", text)
        if unicodedata.category(char) != "Mn"
    )

def tokenize_text(text: str):
    text = text.lower()
    text = remove_accents(text)
    tokens = re.findall(TOKEN_PATTERN, text)

    # remove very short tokens
    tokens = [t for t in tokens if len(t) > 2]

    return tokens

# processing
rows = []

nlp_clean_files = sorted(NLP_CLEAN_ROOT.rglob("*.txt"))

for txt_path in nlp_clean_files:
    rel_path = txt_path.relative_to(NLP_CLEAN_ROOT)
    out_path = TOKENS_ROOT / rel_path.with_suffix(".txt")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    tokens = tokenize_text(text)

    # save tokens (one per line)
    out_path.write_text("\n".join(tokens), encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "num_tokens": len(tokens),
        "num_unique_tokens": len(set(tokens)),
        "avg_token_length": sum(len(t) for t in tokens) / len(tokens) if tokens else 0,
        "status": "ok"
    })

df_tokens = pd.DataFrame(rows)
df_tokens.to_csv(TOKENS_MANIFEST_PATH, index=False, encoding="utf-8")

print("files tokenized:", len(df_tokens))
print("manifest saved to:", TOKENS_MANIFEST_PATH)

display(df_tokens[["num_tokens", "num_unique_tokens", "avg_token_length"]].describe())

files tokenized: 333
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_tokens.csv


,num_tokens,num_unique_tokens,avg_token_length
count,333.000000,333.000000,333.000000
mean,1001.576577,317.813814,7.274005
std,1741.328524,364.423337,0.293638
min,23.000000,20.000000,6.228070
25%,134.000000,89.000000,7.085821
50%,350.000000,164.000000,7.291667
75%,1066.000000,410.000000,7.474415
max,10618.000000,2035.000000,8.204688


### 5. Stopword Removal
This stage removes general Spanish and legal-specific stopwords from the tokenized corpus to reduce noise and retain more meaningful terms. It preserves the folder structure, outputs cleaned tokens, and generates a manifest with statistics to track the filtering process.

In [8]:
# input from previous stage
TOKENS_ROOT = CLEAN_ROOT / "tokens"

# output
STOPWORDS_ROOT = CLEAN_ROOT / "tokens_no_stopwords"
STOPWORDS_ROOT.mkdir(parents=True, exist_ok=True)

STOPWORDS_MANIFEST_PATH = MANIFEST_ROOT / "manifest_stopwords.csv"

# general spanish stopwords (normalized: lowercase, no accents)
SPANISH_STOPWORDS = {
    "de", "del", "la", "las", "el", "los", "un", "una", "unos", "unas",
    "y", "o", "u", "e", "en", "por", "para", "con", "sin", "sobre", "entre",
    "a", "al", "ante", "bajo", "cabe", "contra", "desde", "durante", "hasta",
    "hacia", "mediante", "segun", "tras", "versus", "via",
    "que", "se", "su", "sus", "lo", "le", "les", "como", "mas",
    "ya", "no", "si", "tambien", "solo",
    "este", "esta", "estos", "estas", "ese", "esa", "esos", "esas",
    "aquel", "aquella", "aquellos", "aquellas",
    "mi", "mis", "tu", "tus", "nuestro", "nuestra", "nuestros", "nuestras",
    "haber", "ha", "han", "haya", "hay", "fue", "fueron", "ser", "es", "son",
    "era", "eran", "sea", "sido", "siendo", "estar", "esta", "estan",
    "tiene", "tienen", "tener", "podra", "podran",
    "debe", "deben", "debera", "deberan"
}

# legal / structural stopwords (normalized)
LEGAL_STOPWORDS = {
    "decreto", "decretos",
    "articulo", "articulos",
    "ley", "leyes",
    "paragrafo", "paragrafos",
    "capitulo", "capitulos",
    "titulo", "titulos",
    "seccion", "secciones",
    "parte", "partes",
    "libro", "libros",
    "numeral", "numerales",
    "inciso", "incisos",
    "presente", "dispuesto", "disposiciones",
    "reglamentario", "reglamentaria", "reglamentarias", "reglamentarios",
    "republica", "colombia",
    "ministerio", "ministerios",
    "nacional", "nacionales",
    "vigencia", "publicacion",
    "objeto", "ambito", "aplicacion"
}

# additional low-semantic stopwords (discourse / filler terms)
EXTRA_STOPWORDS = {
    "cual", "cuales", "cada", "uno", "una",
    "mismo", "misma", "mismos", "mismas",
    "dentro", "respecto", "forma", "general",
    "asi", "tambien", "siguiente", "siguientes",
    # new additions
    "unico", "texto", "efectos", "dicho", "numero"
}

STOPWORDS = SPANISH_STOPWORDS.union(LEGAL_STOPWORDS).union(EXTRA_STOPWORDS)

def remove_stopwords(tokens):
    return [tok for tok in tokens if tok not in STOPWORDS]

# processing
rows = []

token_files = sorted(TOKENS_ROOT.rglob("*.txt"))

for txt_path in token_files:
    rel_path = txt_path.relative_to(TOKENS_ROOT)
    out_path = STOPWORDS_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tokens = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    tokens_clean = remove_stopwords(tokens)

    # save tokens (one per line)
    out_path.write_text("\n".join(tokens_clean), encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "tokens_before": len(tokens),
        "tokens_after": len(tokens_clean),
        "tokens_removed": len(tokens) - len(tokens_clean),
        "unique_before": len(set(tokens)),
        "unique_after": len(set(tokens_clean)),
        "status": "ok"
    })

df_stopwords = pd.DataFrame(rows)
df_stopwords.to_csv(STOPWORDS_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_stopwords))
print("manifest saved to:", STOPWORDS_MANIFEST_PATH)

display(
    df_stopwords[
        ["tokens_before", "tokens_after", "tokens_removed", "unique_before", "unique_after"]
    ].describe()
)

files processed: 333
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_stopwords.csv


,tokens_before,tokens_after,tokens_removed,unique_before,unique_after
count,333.000000,333.000000,333.000000,333.000000,333.000000
mean,1001.576577,702.102102,299.474474,317.813814,280.492492
std,1741.328524,1243.287498,500.107040,364.423337,344.519874
min,23.000000,13.000000,10.000000,20.000000,12.000000
25%,134.000000,91.000000,45.000000,89.000000,71.000000
50%,350.000000,239.000000,117.000000,164.000000,130.000000
75%,1066.000000,737.000000,320.000000,410.000000,353.000000
max,10618.000000,7515.000000,3167.000000,2035.000000,1941.000000


### 6. Lemmatization
This stage applies lemmatization to reduce tokens to their base forms, improving consistency and reducing vocabulary size. It preserves structure, outputs normalized tokens, and generates a manifest for traceability.

In [9]:
# input from previous stage
STOPWORDS_ROOT = CLEAN_ROOT / "tokens_no_stopwords"

# load spacy model (disable unused components for efficiency)
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

# output
LEMMAS_ROOT = CLEAN_ROOT / "tokens_lemmas"
LEMMAS_ROOT.mkdir(parents=True, exist_ok=True)

LEMMAS_MANIFEST_PATH = MANIFEST_ROOT / "manifest_lemmas.csv"

# keep valid domain-specific terms even if they look unusual
VALID_DOMAIN_TERMS = {
    "dian", "niif", "iama", "ain", "tasa", "abogacia"
}

# manual corrections for frequent domain-specific cases
LEMMA_CORRECTIONS = {
    "diar": "dian",
    "iama": "iamas",
    "autopart": "autoparte",
    "ventar": "venta",
    "abogaciar": "abogacia",
    "permanenciar": "permanencia",
    "diferenciabl": "diferenciable",
    "tecnologiar": "tecnologia",
    "senalir": "senalar",
    "adoptarar": "adoptar",
    "regimar": "regimen",
    "coexistencio": "coexistencia",
    "ensamblir": "ensamblar",
    "trav": "traves",
    "directiir": "directivo",
    "controlant": "controlante",
    "aduan": "aduana",
    "ivo": "iva"
}

# remove recurrent invalid lemmas produced by spaCy
INVALID_LEMMAS = {
    "sero", "sera", "dema", "areo", "pai", "jo", "aan",
    "abriro", "abrirar", "abogaciar"
}

import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

import src_python.correction_dictionary as correction_dictionary
importlib.reload(correction_dictionary)

MALFORMED_LEMMA_CORRECTIONS = correction_dictionary.MALFORMED_LEMMA_CORRECTIONS

print("corrections loaded:", len(MALFORMED_LEMMA_CORRECTIONS))

def normalize_malformed_lemma(lemma: str) -> str:
    return MALFORMED_LEMMA_CORRECTIONS.get(lemma, lemma)
    
    
    if lemma.endswith ("bl") and len (lemma) > 4:
        return lemma + "e"

    if lemma.endswith("ant") and len(lemma) > 4:
        return lemma + "e"

    # fix malformed -izar verbs  fix malformed -izar verbs
    if lemma.endswith("icir") and len(lemma) > 5:
        return lemma [:-4] + "izar"
    
    return lemma
    
def lemmatize_tokens(token_list):
    doc = nlp(" ".join(token_list))
    lemmas = []

    for token in doc:
        lemma = token.lemma_.strip()
        if not lemma:
            continue

        # normalize lemma
        lemma = remove_accents(lemma.lower())

        # split malformed multi-word lemmas returned by spaCy
        lemma_parts = [part.strip() for part in lemma.split() if part.strip()]

        for part in lemma_parts:
            # manual corrections
            part = LEMMA_CORRECTIONS.get(part, part)
            part = normalize_malformed_lemma(part)

            # keep valid domain-specific terms
            if part in VALID_DOMAIN_TERMS:
                lemmas.append(part)
                continue

            # remove stopwords reintroduced by lemmatization
            if part in STOPWORDS:
                continue

            # remove known invalid lemmas
            if part in INVALID_LEMMAS:
                continue

            # keep only valid lemmas
            if is_valid_token(part):
                lemmas.append(part)

    return lemmas
    
# read files recursively
token_files = sorted(STOPWORDS_ROOT.rglob("*.txt"))
print("files found:", len(token_files))

rows = []

for txt_path in token_files:
    rel_path = txt_path.relative_to(STOPWORDS_ROOT)
    out_path = LEMMAS_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tokens = txt_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    lemmas = lemmatize_tokens(tokens)

    # save lemmas (one per line)
    out_path.write_text("\n".join(lemmas), encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "tokens_before": len(tokens),
        "tokens_after": len(lemmas),
        "unique_before": len(set(tokens)),
        "unique_after": len(set(lemmas)),
        "status": "ok"
    })

df_lemmas = pd.DataFrame(rows)
df_lemmas.to_csv(LEMMAS_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_lemmas))
print("manifest saved to:", LEMMAS_MANIFEST_PATH)
print("columns:", df_lemmas.columns.tolist())
print("shape:", df_lemmas.shape)

if not df_lemmas.empty:
    display(
        df_lemmas[
            ["tokens_before", "tokens_after", "unique_before", "unique_after"]
        ].describe()
    )
else:
    print("df_lemmas is empty. check input path.")

C:\Users\dtruj\Documents\venv_nlp\Lib\site-packages\spacy\util.py:969: UserWarning: [W095] Model 'es_core_news_sm' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


corrections loaded: 221
files found: 333
files processed: 333
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_lemmas.csv
columns: ['input_file', 'output_file', 'rel_path', 'tokens_before', 'tokens_after', 'unique_before', 'unique_after', 'status']
shape: (333, 8)


,tokens_before,tokens_after,unique_before,unique_after
count,333.000000,333.000000,333.000000,333.000000
mean,702.102102,683.357357,280.492492,240.204204
std,1243.287498,1210.644569,344.519874,274.036604
min,13.000000,13.000000,12.000000,12.000000
25%,91.000000,89.000000,71.000000,67.000000
50%,239.000000,232.000000,130.000000,118.000000
75%,737.000000,720.000000,353.000000,310.000000
max,7515.000000,7360.000000,1941.000000,1488.000000


#### Vocabulary Validation
This step evaluates the quality of the final vocabulary after preprocessing and lemmatization, ensuring that the tokens are semantically meaningful, free of noise, and representative of the legal content before proceeding to modeling.

In [10]:
from collections import Counter

all_tokens = []

for file in LEMMAS_ROOT.rglob("*.txt"):
    tokens = file.read_text().splitlines()
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

freq.most_common(60)

[('comercio', 1957),
 ('turismo', 1786),
 ('importacion', 1495),
 ('cuando', 1406),
 ('servicio', 1344),
 ('industria', 1301),
 ('programa', 1276),
 ('informacion', 1223),
 ('bien', 1206),
 ('establecido', 1154),
 ('caso', 1142),
 ('sociedad', 1096),
 ('especial', 991),
 ('proceso', 982),
 ('administrativo', 981),
 ('solicitud', 956),
 ('termino', 950),
 ('entidad', 922),
 ('exportacion', 914),
 ('producto', 889),
 ('actividad', 871),
 ('acuerdo', 866),
 ('direccion', 848),
 ('persona', 845),
 ('otro', 843),
 ('registro', 843),
 ('valor', 839),
 ('ano', 787),
 ('norma', 768),
 ('requisito', 763),
 ('zona', 761),
 ('autoridad', 753),
 ('sistema', 739),
 ('previsto', 728),
 ('desarrollo', 727),
 ('conformidad', 714),
 ('sector', 709),
 ('fecha', 707),
 ('tecnico', 702),
 ('superintendencia', 701),
 ('usuario', 682),
 ('turistico', 648),
 ('cumplimiento', 644),
 ('realizar', 635),
 ('todo', 632),
 ('impuesto', 628),
 ('partir', 613),
 ('produccion', 612),
 ('empresa', 604),
 ('aduana', 60

In [11]:
from collections import Counter
import pandas as pd
import re

# collect all lemmas
all_lemmas = []

for file in LEMMAS_ROOT.rglob("*.txt"):
    lemmas = file.read_text(encoding="utf-8", errors="ignore").splitlines()
    all_lemmas.extend(lemmas)

freq = Counter(all_lemmas)

# suspicious lemma patterns
SUSPICIOUS_PATTERNS = [
    r".*ar$",        # possible truncated or malformed infinitives
    r".*ir$",        # possible malformed infinitives
    r".*irir$",      # repeated infinitive ending
    r".*arar$",      # repeated infinitive ending
    r".*erar$",      # malformed future/infinitive-like ending
    r".*enciar$",    # malformed nominal ending
    r".*ciar$",      # possible malformed noun/verb
    r".*ient$",      # missing final e
    r".*r$",         # generic truncated/malformed ending
]

# known examples to track explicitly
KNOWN_SUSPICIOUS = {
    "separ", "ampar", "tendro", "determin", "veintitr", "pequenar",
    "dosciento", "noveciento", "serar", "queir", "aclarir", "seguirar",
    "ruser", "tramit", "angelasr", "aparia", "superintendenciar", "onar",
    "rismo", "porun", "holme", "procederar", "camar", "raa", "tarjeto",
    "goordinacien", "observanciar", "adquir", "aparezcar", "proveniar",
    "fuerir", "hubierir", "tendient", "necesariar", "estir", "amenacir",
    "consecuenciar", "acrecientir", "resultir", "concurrencio", "derivir",
    "polizar", "extraigar", "anir", "lesionir", "colectir"
}

def is_suspicious_lemma(term: str) -> bool:
    if term in KNOWN_SUSPICIOUS:
        return True

    if len(term) <= 3:
        return True

    if any(re.fullmatch(pattern, term) for pattern in SUSPICIOUS_PATTERNS):
        return True

    return False

# build suspicious terms table
df_suspicious_lemmas = pd.DataFrame([
    {"term": term, "frequency": count}
    for term, count in freq.items()
    if is_suspicious_lemma(term)
])

df_suspicious_lemmas = df_suspicious_lemmas.sort_values(
    by="frequency",
    ascending=False
).reset_index(drop=True)

print("suspicious lemmas found:", len(df_suspicious_lemmas))

#display(df_suspicious_lemmas.head(100))

suspicious lemmas found: 1685


In [12]:
suspicious_list = df_suspicious_lemmas["term"].head(100).tolist()

print(suspicious_list)

['valor', 'ano', 'sector', 'realizar', 'partir', 'presentar', 'contar', 'hacer', 'dia', 'establecer', 'anterior', 'auxiliar', 'exterior', 'uso', 'poder', 'modificar', 'aplicar', 'encontrar', 'lugar', 'liquidador', 'determinar', 'considerar', 'quedar', 'adicionar', 'productor', 'prestador', 'solicitar', 'superior', 'regir', 'incluir', 'corresponder', 'desarrollar', 'operador', 'acreditar', 'requerir', 'publicar', 'promotor', 'cometer', 'investigador', 'deudor', 'sustituir', 'inter', 'proceder', 'informar', 'decretar', 'adoptar', 'prorrogar', 'permitir', 'utilizar', 'indicar', 'inferior', 'contener', 'existir', 'consumidor', 'importador', 'referir', 'proveedor', 'declarar', 'pagar', 'reglamentar', 'efectuar', 'pretender', 'exportador', 'entrar', 'obtener', 'garantizar', 'importar', 'administrador', 'verificar', 'mantener', 'similar', 'interventor', 'adelantar', 'ejercer', 'promover', 'factor', 'acreedor', 'iniciar', 'ordenar', 'llevar', 'generar', 'caracter', 'definir', 'director', 'part

In [13]:
suspicious_list = list(
    zip(
        df_suspicious_lemmas["term"].head(100),
        df_suspicious_lemmas["frequency"].head(100)
    )
)

print(suspicious_list)

[('valor', 839), ('ano', 787), ('sector', 709), ('realizar', 635), ('partir', 613), ('presentar', 596), ('contar', 575), ('hacer', 556), ('dia', 520), ('establecer', 493), ('anterior', 475), ('auxiliar', 396), ('exterior', 380), ('uso', 373), ('poder', 371), ('modificar', 369), ('aplicar', 361), ('encontrar', 356), ('lugar', 336), ('liquidador', 315), ('determinar', 301), ('considerar', 301), ('quedar', 296), ('adicionar', 287), ('productor', 277), ('prestador', 274), ('solicitar', 270), ('superior', 269), ('regir', 268), ('incluir', 262), ('corresponder', 259), ('desarrollar', 258), ('operador', 257), ('acreditar', 247), ('requerir', 243), ('publicar', 240), ('promotor', 221), ('cometer', 221), ('investigador', 217), ('deudor', 216), ('sustituir', 212), ('inter', 206), ('proceder', 205), ('informar', 204), ('decretar', 201), ('adoptar', 196), ('prorrogar', 191), ('permitir', 188), ('utilizar', 185), ('indicar', 184), ('inferior', 183), ('contener', 182), ('existir', 182), ('consumidor

In [15]:
suspicious_list = list(
    zip(
        df_suspicious_lemmas["term"].head(100),
        df_suspicious_lemmas["frequency"].head(100)
    )
)

print(suspicious_list)

[('adicionar', 287), ('declarar', 158), ('iniciar', 121), ('aclarar', 66), ('diligenciar', 42), ('diferenciar', 36), ('adquirir', 36), ('proporcionar', 32), ('seleccionar', 28), ('relacionar', 25), ('evidenciar', 21), ('renunciar', 20), ('preparar', 19), ('funcionar', 15), ('beneficiar', 14), ('pronunciar', 11), ('sancionar', 9), ('financiar', 8), ('mencionar', 8), ('promocionar', 8), ('comerciar', 7), ('anunciar', 6), ('franquiciar', 6), ('solucionar', 5), ('separar', 5), ('referenciar', 4), ('potenciar', 4), ('distanciar', 4), ('negociar', 4), ('propiciar', 4), ('enunciar', 4), ('fiduciar', 4), ('repotenciar', 3), ('perfeccionar', 2), ('denunciar', 2), ('comparar', 2), ('asociar', 2), ('reiniciar', 2), ('deferenciar', 2), ('condicionar', 1), ('presenciar', 1), ('direccionar', 1), ('depreciar', 1), ('gerenciar', 1), ('cofinanciar', 1), ('sonaciar', 1), ('acondicionar', 1), ('amparar', 1)]
